# Flambeau 2-Week Outage Timing - 2027

*Which two-week window in 2027 costs Dairyland the least in foregone Flambeau revenue?*

**Prepared for:** Dairyland Power Cooperative (technical + non-technical reviewers)  
**Author:** _(your name)_  
**Runtime:** ~10 minutes

## TL;DR

- **Recommendation:** _(filled by notebook 03 output)_ - three non-overlapping 14-day windows. Two are picked to minimize expected lost revenue; one is picked to minimize the P90 (bad-year) cost, so the reviewer can see the tradeoff between 'lowest average' and 'lowest bad year.'
- **Reading the problem carefully.** The ticket asks for a window where lost revenue is *consistently* low. 'Consistently' means both a low average across historical years AND low year-to-year variability. Both are reported for every candidate.
- **Timing matters, but not enormously.** The gap between the best and worst window is about `$_____` on an anchor-year annual revenue of about `$_____`; picking any window on the recommended plateau captures most of the saving.
- **Confidence.** Estimates come from 10 complete calendar years of history (2016-2025). Uncertainty is reported as SD across years (year-to-year swing), an empirical 10-90 range (usual good year to usual bad year), and a bootstrap CI on the mean (how much having only 10 years hurts the estimate).
- **What is *not* in the number.** Capacity payments, RECs, and ancillary-service revenue are excluded; only day-ahead energy revenue is counted.

## The problem in one paragraph

Flambeau is a small hydro plant that sells its output into the MISO day-ahead market at a location-specific price (LMP, $/MWh). A planned maintenance outage means the plant produces zero MW for two weeks, forgoing whatever revenue it would have earned during that time. The best window minimizes that forgone revenue. Because both the price (LMP) and the plant's output (MW, driven by river flow) vary seasonally, the answer is not obvious - and because they vary in *opposite* seasonal directions, revenue is flatter than either curve alone.

## Approach (60-second version)

1. **Get the price history.** Download MISO day-ahead ExPost LMP for the DPC.FLAMBEAU node, 2016-01-01 through today. The published files follow a fixed URL pattern (`YYYYMM_da_expost_lmp_csv.zip` before 2023, then daily `YYYYMMDD_da_expost_lmp.csv`) - no page scraping needed.
2. **Fill in the generation.** The provided generation file has one reading every ~4 days. We interpolate it to hourly on a UTC grid; river flow moves slowly, so linear interpolation is defensible. A sensitivity check confirms the choice does not swing the answer.
3. **Compute revenue.** `revenue = MW * LMP` per hour, summed to daily totals by local calendar date.
4. **Rank every window.** For each 2-week start date in 2027, compute the historical 14-day revenue for that calendar span in each of 2016-2025. Convert each to a *share of that year's annual revenue* (so 2022's fuel-price spike does not dominate), average across years, and scale by a recent-year annual anchor to get 2027-relevant dollars.
5. **Uncertainty and honesty check.** Report SD, empirical range, and bootstrap CI. Run leave-one-year-out to check that the winning window is not just noise in the historical sample.

In [ ]:
# Step 1. Imports and paths. Load the three artifacts saved by notebook 03:
#         window_summary.parquet, candidates.parquet, loyo.parquet.


## Headline chart 1 - Price and output are seasonally opposite

LMP peaks in summer (AC load). Flambeau output peaks in spring (snowmelt). The product - revenue - is flatter than either. This is why the answer must come from `MW * LMP`, not from either curve alone.

In [ ]:
# Step 2. Two-panel chart:
#   Panel A: monthly mean LMP (line, band = 10-90 across years) and
#            monthly mean MW on a twin axis - visually anti-correlated.
#   Panel B: monthly mean daily revenue - the product, plainly flatter.


## Headline chart 2 - Cost of every possible 2-week window in 2027

Each point is a candidate outage start date. The shaded band is the historical 10-90 range across the 10-year sample. The winning windows sit on a broad plateau, not a sharp point - which means the exact start date is a soft choice.

In [ ]:
# Step 3. Expected-cost curve with uncertainty band.
#   x = 2027 start date; y = expected cost. Shade 10-90 band.
#   Annotate the 3 candidate windows with vertical markers.


## Recommendation - three candidate windows

Any of these is a good pick; they differ in expected cost by less than the uncertainty of each. Presented in order of expected cost.

In [ ]:
# Step 4. Recommendation table.
#   Columns: Rank, Window (start .. end), Expected cost ($), P90 cost ($),
#            10-90 empirical range ($), Bootstrap 95% CI ($).
#   Format dollars with commas, no cents.
#
# Render as a styled DataFrame so it looks like a slide, not a repr.


## Honesty check - would this recommendation have held up historically?

For each year 2016-2025 we pretend we do not have that year's data, pick the best window from the other nine, and score it on the held-out year. `optimism` is how much larger the actual held-out cost is than our in-sample expected cost. If it is small across years, the pick is robust.

In [ ]:
# Step 5. LOYO table + one summary sentence:
#   'Mean optimism across held-out years: $X (Y% of the expected cost).'


## What could make this wrong

- **Price level change.** If 2027 LMPs are systematically different from 2023-2025, the dollar figures scale accordingly. The *ranking* of windows does not change (the anchor is a multiplicative constant); only the magnitudes do.
- **Hydrology shift.** A dry (or wet) summer in 2027 would move MW away from the historical mean. Our uncertainty band captures year-to-year variation in the last decade but not a step change from that regime.
- **Within-day price shaping.** A flat within-day MW profile from interpolation removes any hour-of-day arbitrage Flambeau performs. If the plant actively shapes output toward peak-price hours, we understate revenue (and therefore lost revenue). Direction is known, magnitude is not.
- **Non-energy revenue.** Capacity payments, RECs, and ancillary services are out of scope. Add-ons would raise, not lower, the true cost of any outage.

## Appendix - reproducibility

- Environment: `requirements.txt` in repo root.
- Full pipeline: `notebooks/01_acquisition.ipynb`, `02_eda.ipynb`, `03_windows.ipynb`.
- Library code: `src/miso_lmp.py`, `src/gen.py`, `src/windows.py`.
- Data cache: `data/processed/` (parquet). First run downloads MISO archives (~30-60 min); re-runs are instant.